# 第107章 共享单车需求预测项目

使用 UCI Bike Sharing 小时数据，按照时间序列回归的教学流程预测共享单车小时需求。

## 项目背景

根据日历与天气信息预测全网下一时段的租赁需求。本章关注时间回归建模，不使用站点库存和OD信息，也不把相关性解释为天气的因果效应。

## 学习目标

- 审计时间索引和目标构成
- 识别目标组成字段造成的直接泄漏
- 构造周期时间特征
- 使用时间顺序划分和季节基线
- 比较随机森林与梯度提升
- 通过残差切片和特征重要性理解模型


## 数据字典

| 字段 | 含义 | 使用说明 |
| --- | --- | --- |
| timestamp | 日期与小时 | 排序和切分依据 |
| workingday/weathersit | 工作日/天气 | 已知场景特征 |
| temp/hum/windspeed | 气象变量 | 归一化连续变量 |
| casual/registered | 需求组成 | 直接构成目标，禁止作为特征 |
| cnt | 总租赁量 | 回归目标 |

## 数据质量检查清单

- 时间重复、排序与缺失小时
- cnt是否等于casual加registered
- 缺失值和变量范围
- 训练时间严格早于测试
- 目标组成字段泄漏
- 总体误差之外的时段差异


## 项目任务

1. 明确小时预测目标和预测提前量
2. 审计时间索引与目标构成
3. 识别并排除泄漏字段
4. 探索工作日、小时和天气差异
5. 构造小时与月份周期特征
6. 按时间划分训练、验证和测试
7. 建立季节基线并比较两个模型
8. 报告MAE、RMSE、R²与峰值识别
9. 分析残差切片和高误差时段
10. 解释特征重要性并总结局限


## 项目交付物

- 一份从数据审计到模型评价可完整运行的 Notebook
- 数据清洗前后样本变化和关键质量检查结果
- 基线与候选模型的指标对比表
- 错误切片、特征解释和有边界的业务结论

## 阶段检查点

- [ ] 数据与目标定义完成：样本粒度、预测时点和指标已写清楚
- [ ] 基线完成：知道复杂模型相对什么标准比较
- [ ] 模型评价完成：测试集只使用一次，并检查误差切片
- [ ] 交付完成：结论与证据对应，不把相关性写成因果

## 最低完成标准

- 每个代码阶段都有可见输出，不能依赖未展示的隐藏状态。
- 所有关键清洗、筛选和评价口径都写在 Markdown 或注释中。
- 最终结论至少引用一个数值或图表证据，并说明适用范围。

## 提升任务

完成基础验收后，可以增加一个对照方案、一个分组切片或一个参数敏感性实验，比较结果是否稳定。


## 1. 时间索引与数据质量审计

组合日期和小时形成真正的时间主键，并检查排序、重复和缺失。


In [ ]:
import numpy as np
import pandas as pd
from js import window
df=pd.read_csv(f"{window.location.origin}/datasets/bike_sharing_hour.csv",parse_dates=['dteday'])
df['timestamp']=df.dteday+pd.to_timedelta(df.hr,unit='h'); df=df.sort_values('timestamp').reset_index(drop=True)
audit=pd.Series({'行数':len(df),'时间重复':df.timestamp.duplicated().sum(),'缺失值':df.isna().sum().sum(),'时间是否递增':df.timestamp.is_monotonic_increasing})
print(audit.to_string()); print('时间范围:',df.timestamp.min(),'至',df.timestamp.max())


## 2. 目标构成与泄漏检查

casual和registered相加就是cnt，若作为特征会让模型提前看到答案。


In [ ]:
composition_error=(df.cnt!=df.casual+df.registered).sum(); forbidden=['casual','registered','cnt']
print('目标构成错误:',composition_error); print('禁止进入特征:',forbidden); display(df[['cnt','casual','registered']].describe().round(1))


## 3. 探索小时、工作日与天气场景

探索用于认识数据和设计误差切片，不把组间差异直接解释为因果作用。


In [ ]:
hour_profile=df.groupby(['workingday','hr']).cnt.mean(); weather_profile=df.groupby('weathersit').agg(hours=('cnt','size'),mean_demand=('cnt','mean'),p90=('cnt',lambda x:x.quantile(.9)))
print('工作日需求最高小时:\n',hour_profile.loc[1].nlargest(5).round(1)); print('非工作日需求最高小时:\n',hour_profile.loc[0].nlargest(5).round(1)); display(weather_profile.round(1))


## 4. 构造周期时间特征

正余弦编码让23点和0点、12月和1月在特征空间中保持相邻。


In [ ]:
df['hour_sin']=np.sin(2*np.pi*df.hr/24); df['hour_cos']=np.cos(2*np.pi*df.hr/24); df['month_sin']=np.sin(2*np.pi*df.mnth/12); df['month_cos']=np.cos(2*np.pi*df.mnth/12)
features=['season','yr','mnth','hr','holiday','weekday','workingday','weathersit','temp','atemp','hum','windspeed','hour_sin','hour_cos','month_sin','month_cos']
assert not set(features)&set(forbidden); print('特征数量:',len(features)); display(df[['hr','hour_sin','hour_cos','mnth','month_sin','month_cos']].head())


## 5. 时间划分与季节基线

训练、验证和测试严格按时间排列；工作日×小时均值构成直观基线。


In [ ]:
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score
train_end=int(len(df)*.64); val_end=int(len(df)*.80); train=df.iloc[:train_end]; val=df.iloc[train_end:val_end]; test=df.iloc[val_end:]
baseline_profile=train.groupby(['workingday','hr']).cnt.mean(); val_baseline=np.array([baseline_profile.get((w,h),train.cnt.mean()) for w,h in zip(val.workingday,val.hr)])
print('训练/验证/测试:',len(train),len(val),len(test)); print('训练截止:',train.timestamp.max(),'测试开始:',test.timestamp.min()); print('验证集季节基线MAE:',round(mean_absolute_error(val.cnt,val_baseline),1))


## 6. 比较随机森林与梯度提升

在验证时段上比较两个非线性回归模型，最终测试时段保持独立。


In [ ]:
from sklearn.ensemble import RandomForestRegressor,HistGradientBoostingRegressor
models={'随机森林':RandomForestRegressor(n_estimators=180,min_samples_leaf=3,n_jobs=-1,random_state=107),'梯度提升':HistGradientBoostingRegressor(max_iter=220,max_leaf_nodes=20,l2_regularization=1,random_state=107)}
rows=[]
for name,model in models.items():
    model.fit(train[features],train.cnt); rows.append([name,mean_absolute_error(val.cnt,model.predict(val[features]))])
validation=pd.DataFrame(rows,columns=['model','validation_MAE']).sort_values('validation_MAE'); display(validation.round(1))
best_name=validation.iloc[0].model; dev=df.iloc[:val_end]; best_model=models[best_name].fit(dev[features],dev.cnt); prediction=np.maximum(0,best_model.predict(test[features]))


## 7. 最终回归指标与峰值识别

总体回归误差之外，再检查模型是否识别出真实高需求时段。


In [ ]:
test_profile=dev.groupby(['workingday','hr']).cnt.mean(); test_baseline=np.array([test_profile.get((w,h),dev.cnt.mean()) for w,h in zip(test.workingday,test.hr)])
metrics=pd.Series({'MAE':mean_absolute_error(test.cnt,prediction),'RMSE':mean_squared_error(test.cnt,prediction)**.5,'R2':r2_score(test.cnt,prediction),'Baseline_MAE':mean_absolute_error(test.cnt,test_baseline)})
peak_threshold=dev.cnt.quantile(.9); actual_peak=test.cnt>=peak_threshold; predicted_peak=prediction>=peak_threshold
peak_precision=((actual_peak)&(predicted_peak)).sum()/max(predicted_peak.sum(),1); peak_recall=((actual_peak)&(predicted_peak)).sum()/max(actual_peak.sum(),1)
print('最佳模型:',best_name); print(metrics.round(2).to_string()); print('高需求阈值:',round(peak_threshold,1),'峰值Precision:',round(peak_precision,3),'峰值Recall:',round(peak_recall,3))


## 8. 构造残差并进行时间切片

总体MAE可能掩盖特定小时、月份或工作日场景中的系统性误差。


In [ ]:
errors=test[['timestamp','hr','mnth','workingday','weathersit','cnt']].copy(); errors['prediction']=prediction; errors['residual']=errors.cnt-errors.prediction; errors['absolute_error']=errors.residual.abs()
hour_error=errors.groupby('hr').absolute_error.mean().nlargest(6); month_error=errors.groupby('mnth').absolute_error.mean().nlargest(4); workday_error=errors.groupby('workingday').absolute_error.agg(['count','mean','median'])
print('误差最高小时:\n',hour_error.round(1)); print('误差最高月份:\n',month_error.round(1)); print('工作日切片:\n',workday_error.round(1))


## 9. 检查高误差案例

查看最大正负残差，判断模型在节假日、异常天气或需求突变时如何失效。


In [ ]:
largest_under=errors.nlargest(6,'residual')[['timestamp','cnt','prediction','residual','workingday','weathersit']]
largest_over=errors.nsmallest(6,'residual')[['timestamp','cnt','prediction','residual','workingday','weathersit']]
print('明显低估案例:'); display(largest_under.round(1)); print('明显高估案例:'); display(largest_over.round(1))


## 10. 特征解释与模型局限

置换重要性展示模型依赖的预测信号，并明确数据只能支持全网小时需求建模。


In [ ]:
from sklearn.inspection import permutation_importance
sample_n=min(3000,len(test)); sample_idx=np.linspace(0,len(test)-1,sample_n,dtype=int)
permutation=permutation_importance(best_model,test.iloc[sample_idx][features],test.iloc[sample_idx].cnt,n_repeats=3,scoring='neg_mean_absolute_error',random_state=107,n_jobs=-1)
importance=pd.Series(permutation.importances_mean,index=features).sort_values(ascending=False)
print('置换重要性前10:\n',importance.head(10).round(2)); print('局限: 数据没有站点库存、OD流向和未来天气预报，不能据此完成站点级调度或因果解释。')


## 结论与表达

- 时间数据不能随机打乱后评价未来表现
- 目标组成字段是最直接的数据泄漏
- 季节基线能判断复杂模型是否真正提供增益
- 总体误差必须结合峰值和时间切片理解


## 项目验收清单

- 完成时间索引和目标构成审计
- 排除casual与registered
- 使用严格时间三段划分
- 比较季节基线和两个模型
- 报告回归指标、峰值指标和误差切片
- 完成高误差案例与置换重要性分析

建议重新启动内核后从第一个代码单元格运行，确认项目不依赖隐藏状态。


## 本章小结

使用 UCI Bike Sharing 小时数据，按照时间序列回归的教学流程预测共享单车小时需求。


### 你已经完成

- 审计时间索引和目标构成
- 识别目标组成字段造成的直接泄漏
- 构造周期时间特征
- 使用时间顺序划分和季节基线
- 比较随机森林与梯度提升
- 通过残差切片和特征重要性理解模型


### 建模流程速查

| 阶段 | 学习内容 |
| --- | --- |
| 步骤 1 | 明确小时预测目标和预测提前量 |
| 步骤 2 | 审计时间索引与目标构成 |
| 步骤 3 | 识别并排除泄漏字段 |
| 步骤 4 | 探索工作日、小时和天气差异 |
| 步骤 5 | 构造小时与月份周期特征 |
| 步骤 6 | 按时间划分训练、验证和测试 |
| 步骤 7 | 建立季节基线并比较两个模型 |
| 步骤 8 | 报告MAE、RMSE、R²与峰值识别 |
| 步骤 9 | 分析残差切片和高误差时段 |
| 步骤 10 | 解释特征重要性并总结局限 |


### 质量与结论提醒

- 时间重复、排序与缺失小时
- cnt是否等于casual加registered
- 缺失值和变量范围
- 时间数据不能随机打乱后评价未来表现
- 目标组成字段是最直接的数据泄漏
- 季节基线能判断复杂模型是否真正提供增益
- 总体误差必须结合峰值和时间切片理解


### 学习检查

- [ ] 完成时间索引和目标构成审计
- [ ] 排除casual与registered
- [ ] 使用严格时间三段划分
- [ ] 比较季节基线和两个模型
- [ ] 报告回归指标、峰值指标和误差切片
- [ ] 完成高误差案例与置换重要性分析


### 后续迭代建议

完成验收后，记录一个最值得继续验证的假设：可以是更多数据、不同时间窗口、另一种模型，或一个更细的分组分析。
